# Nonlinear Cart-Pole: Solver, LQR, Neural ODE, PINN, PPO

공통 상태는 $x(t)=(p(t),\theta(t),\dot p(t),\dot\theta(t))\in\mathbb R^4$, 제어 입력은 $u(t)\in\mathbb R$입니다.

- DOP853 high-accuracy reference
- LQR baseline
- Neural ODE vector field learning
- PINN closed-loop trajectory learning
- PPO control


In [1]:
import sys, pathlib, subprocess
if pathlib.Path('/content').exists():
    if not pathlib.Path('/content/RL_tmp').exists():
        subprocess.run(['git','clone','https://github.com/HisameOgasahara/RL_tmp.git','/content/RL_tmp'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','/content/RL_tmp/requirements.txt'], check=True)
    sys.path.insert(0, '/content/RL_tmp/src')
else:
    sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import numpy as np
import torch
import matplotlib.pyplot as plt
from rl_tmp.cartpole_system import CartPoleConfig
from rl_tmp.cartpole_solver import solve_cartpole_reference, make_lqr_controller
from rl_tmp.cartpole_neural_ode import train_cartpole_neural_ode, rollout_learned_field
from rl_tmp.cartpole_pinn import train_cartpole_pinn, predict_cartpole_pinn
from rl_tmp.cartpole_rl import train_cartpole_ppo, rollout_cartpole_policy
from rl_tmp.cartpole_visualize import plot_state_trajectories, plot_training
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


device: cpu


## 1. DOP853 reference와 LQR baseline

비선형 Cart-Pole을 `solve_ivp(method='DOP853')`로 적분하고, 평형점에서 선형화한 $\dot x=Ax+Bu$에 대해 $u=-Kx$ LQR을 구합니다.

In [2]:
cfg = CartPoleConfig()
initial_state = np.array([0.0, 0.2, 0.0, 0.0])
times_5s = np.linspace(0.0, 5.0, 251)
uncontrolled = solve_cartpole_reference(initial_state, times_5s, cfg)
lqr_control, K, A, B = make_lqr_controller(cfg)
lqr_trajectory = solve_cartpole_reference(initial_state, times_5s, cfg, lqr_control)
print('LQR K =', K)
print('open-loop eigenvalues =', np.linalg.eigvals(A))
print('closed-loop eigenvalues =', np.linalg.eigvals(A-B@K))
print('uncontrolled final norm =', np.linalg.norm(uncontrolled[-1]))
print('LQR final norm =', np.linalg.norm(lqr_trajectory[-1]))
plot_state_trajectories(times_5s, uncontrolled, lqr_trajectory, ('uncontrolled DOP853','LQR + DOP853'), 'Cart-Pole: uncontrolled vs LQR')


LQR K = [[ -2.5819889  -45.8462856   -5.0144165  -12.09511023]]
open-loop eigenvalues = [ 0.          0.          3.97185218 -3.97185218]
closed-loop eigenvalues = [-7.48265059+0.j         -2.6613792 +0.j         -1.33200893+0.29190232j -1.33200893-0.29190232j]
uncontrolled final norm = 6.7689488234597235
LQR final norm = 0.019255141785996095


## 2. Neural ODE

DOP853 trajectory에서 상태와 물리 vector field를 학습하고, 학습된 $f_\theta:\mathbb R^4\to\mathbb R^4$를 RK4로 적분해 검증합니다.

In [3]:
neural_result = train_cartpole_neural_ode(cfg, steps=1600, seed=0, device=DEVICE)
times_1s = np.linspace(0.0, 1.0, 41)
neural_initial = np.array([0.1, 0.12, 0.0, 0.0])
neural_reference = solve_cartpole_reference(neural_initial, times_1s, cfg)
neural_prediction = rollout_learned_field(neural_result.model, neural_initial, times_1s, DEVICE).numpy()
print('field MSE: first -> last =', neural_result.history['field_mse'][0], '->', neural_result.history['field_mse'][-1])
print('trajectory MSE =', np.mean((neural_prediction-neural_reference)**2))
print('final-state error =', np.linalg.norm(neural_prediction[-1]-neural_reference[-1]))
plot_training(neural_result.history, 'Neural ODE field training')
plot_state_trajectories(times_1s, neural_reference, neural_prediction, ('DOP853','Neural ODE'), 'Neural ODE rollout')


field MSE: first -> last = 28.405162811279297 -> 0.002427640836685896
trajectory MSE = 0.00019965446249270118
final-state error = 0.08842747350909097


## 3. PINN

LQR closed-loop ODE $\dot x=f(x,-Kx)$ residual을 학습합니다. 초기조건은 $x_\theta(t)=x_0+(t/T)N_\theta(t/T)$로 hard constraint 처리합니다.

In [4]:
pinn_times = np.linspace(0.0, 1.0, 61)
pinn_reference = solve_cartpole_reference(initial_state, pinn_times, cfg, lqr_control)
pinn_result = train_cartpole_pinn(initial_state, cfg, K, t_final=1.0, steps=1800, collocation_points=128, seed=0, device=DEVICE)
pinn_prediction = predict_cartpole_pinn(pinn_result.model, pinn_times, DEVICE).numpy()
print('PINN residual: first -> last =', pinn_result.history['residual'][0], '->', pinn_result.history['residual'][-1])
print('trajectory MSE =', np.mean((pinn_prediction-pinn_reference)**2))
print('final-state error =', np.linalg.norm(pinn_prediction[-1]-pinn_reference[-1]))
plot_training(pinn_result.history, 'Cart-Pole PINN residual')
plot_state_trajectories(pinn_times, pinn_reference, pinn_prediction, ('LQR + DOP853','PINN'), 'PINN closed-loop trajectory')


PINN residual: first -> last = 45.664283752441406 -> 0.04554677754640579
trajectory MSE = 0.028269031134311028
final-state error = 0.28001711849940325


## 4. PPO control

정책은 $\pi_\theta:\mathbb R^4\to\Delta(\{-10,+10\})$이고, $|p|\le2.4$, $|\theta|\le0.5$ 범위에서 upright 유지하도록 학습합니다.

In [5]:
ppo_result = train_cartpole_ppo(cfg, updates=80, batch_size=96, horizon=96, seed=0, device=DEVICE)
ppo_states, ppo_forces, ppo_success = rollout_cartpole_policy(ppo_result.model, initial_state, cfg, t_final=5.0, dt=0.02, device=DEVICE)
ppo_times = np.arange(len(ppo_states))*0.02
print('mean reward: first -> last =', ppo_result.history['mean_reward'][0], '->', ppo_result.history['mean_reward'][-1])
print('failure rate: first -> last =', ppo_result.history['failure_rate'][0], '->', ppo_result.history['failure_rate'][-1])
print('5 s rollout success =', ppo_success)
print('max |theta| during rollout =', np.max(np.abs(ppo_states[:,1])))
print('final state =', ppo_states[-1])
plot_training(ppo_result.history, 'PPO training')
plt.figure(figsize=(8,4)); plt.plot(ppo_times, ppo_states[:,1]); plt.axhline(0.5, linestyle='--'); plt.axhline(-0.5, linestyle='--'); plt.xlabel('t'); plt.ylabel('theta'); plt.title('PPO closed-loop pole angle'); plt.grid(alpha=.2); plt.show()


mean reward: first -> last = 0.8036315441131592 -> 0.8437251448631287
failure rate: first -> last = 0.0342881940305233 -> 0.009765625
5 s rollout success = True
max |theta| during rollout = 0.34259298
final state = [ 1.6538898   0.0531133   0.75193286 -0.54329264]


## 실행 결과 해석

- DOP853를 trajectory 검증 기준으로 사용합니다.
- LQR은 upright 주변의 강한 baseline입니다.
- Neural ODE는 1초 구간에서 reference를 잘 재현했습니다.
- PINN은 더 어려웠고 hard initial condition 적용 후 오차가 줄었습니다.
- PPO는 연속 LQR보다 거친 이산 force만 사용하지만 5초 rollout에 성공했습니다.